# PRAGMA · A‑E(−1) v1.4 — una sola intervención (H2) · un clic

> **No tienes que decidir ni mirar nada.** Arrastra la foto `P1070614.JPG` al panel **Archivos**
> (icono de carpeta, a la izquierda) y pulsa **Entorno de ejecución → Ejecutar todas**. Al final se
> descarga un ZIP: **adjúntalo solo a Claude**.

Qué hace, sin intervención:

1. Instala SAM 2 **en el commit exacto** `2b90b9f5` y descarga SAM 2.1 Large. **Se detiene** si el
   commit o el checkpoint no son los congelados.
2. Encuentra la foto por su huella SHA‑256 y **se detiene** si la luma de cualquiera de los prompts
   se desvía más de 3,0 de lo registrado.
3. Ejecuta las **22 llamadas** del prerregistro: las semillas, la cadena ganadora de v1.3 (que
   debe salir idéntica), la misma cadena con el positivo H2 y las perturbaciones de H2.
4. Empaqueta las 24 máscaras con un manifiesto de hashes en
   `PRAGMA_AEM1v14_<run>_PENDING_EXTERNAL_AUDIT.zip`.

**Por qué no enseña resultados:** la auditoría es **ciega y a doble llave**. No compartas capturas de
este cuaderno con ChatGPT ni con nadie: el paquete ciego lo prepara Claude.

Prerregistro: `aem1/PRERREGISTRO_A-E-menos-1_v1_4.json` (SHA‑256 del archivo `b0cfe33ee6aca8921c3e0c7a01a2604131b5e6cf9a448d6b8635626f13c690aa`;
`content_sha256` `7048b9fabf03544885d749ad3f19d0dd9d373730d8531f87db58e40ca7220b5d`). Auditoría: `auditoria/PROTOCOLO_AUDITORIA_AEM1_v2.md` (rev. 1).

## 0. Antes de pulsar «Ejecutar todas»

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU → L4.** Usa **L4**, como en
   las dos corridas anteriores: la comparación exige que la referencia salga idéntica bit a bit, y
   eso solo está probado en L4.
2. Arrastra `P1070614.JPG` al panel **Archivos**. Si no lo haces, la celda 2 te pedirá la foto con
   un botón **Elegir archivos**. El nombre no importa: se reconoce por su huella.
3. **Entorno de ejecución → Ejecutar todas.** Tarda unos minutos: casi todo es instalar SAM 2 y
   descargar 857 MiB. Al terminar, el navegador descarga el ZIP.

Si aparece un error en rojo, copia el texto y pégalo a Claude. No cambies nada del cuaderno.

In [ ]:
# Celda 1 · SAM 2 en el commit congelado y checkpoint verificado (red; en el arnés local se omite)
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

SAM2_PINNED_COMMIT = "2b90b9f5ceec907a1c18123530e92e794ad901a4"
CHECKPOINT_SHA256_PINNED = "2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318"
CHECKPOINT_BYTES_PINNED = 898083611
REPO_DIR = Path("/content/pragma_sam2_2b90b9f5")
WORK_DIR = Path("/content/pragma_run")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT = CHECKPOINT_DIR / "sam2.1_hiera_large.pt"
CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def run_checked(args, **kwargs):
    print("$", " ".join(map(str, args)))
    return subprocess.run(args, check=True, text=True, **kwargs)

if not (REPO_DIR / ".git").exists():
    # Clon sin blobs y checkout del commit exacto (no «main», que puede moverse).
    run_checked(["git", "clone", "--filter=blob:none", "--no-checkout",
                 "https://github.com/facebookresearch/sam2.git", str(REPO_DIR)])
run_checked(["git", "-C", str(REPO_DIR), "checkout", "--quiet", SAM2_PINNED_COMMIT])
SAM2_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if SAM2_COMMIT != SAM2_PINNED_COMMIT:
    raise RuntimeError(f"FAIL_FREEZE: SAM 2 está en {SAM2_COMMIT}, no en {SAM2_PINNED_COMMIT}. No se genera nada.")

install_env = os.environ.copy()
install_env["SAM2_BUILD_CUDA"] = "0"
run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR, env=install_env)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import importlib
importlib.invalidate_caches()
import sam2
print("SAM 2 importable:", Path(sam2.__file__).resolve())

def download(url, destination):
    temporary = destination.with_suffix(destination.suffix + ".part")
    def progress(blocks, block_size, total):
        if total > 0 and blocks % 128 == 0:
            print(f"Descarga: {min(100, blocks*block_size*100/total):5.1f}%", end="\r")
    urllib.request.urlretrieve(url, temporary, reporthook=progress)
    temporary.replace(destination)
    print(f"\nCheckpoint: {destination.stat().st_size / 2**20:.1f} MiB")

if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size != CHECKPOINT_BYTES_PINNED:
    download(CHECKPOINT_URL, CHECKPOINT)
digest = hashlib.sha256()
with open(CHECKPOINT, "rb") as handle:
    for block in iter(lambda: handle.read(1 << 20), b""):
        digest.update(block)
if CHECKPOINT.stat().st_size != CHECKPOINT_BYTES_PINNED or digest.hexdigest() != CHECKPOINT_SHA256_PINNED:
    raise RuntimeError("FAIL_FREEZE: el checkpoint no es el congelado (bytes o SHA-256). No se genera nada.")
os.chdir(WORK_DIR)
print("Congelado verificado · SAM 2", SAM2_COMMIT, "· checkpoint", CHECKPOINT_SHA256_PINNED[:12], "…")

In [ ]:
# Celda 2 · hardware, precisión y foto (se reconoce por SHA-256; el nombre puede variar)
EXPECTED_IMAGE_SHA256 = "8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d"
EXPECTED_IMAGE_BYTES = 4260352
import hashlib, io, json, os, platform, time, uuid, zipfile
from contextlib import nullcontext
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps
import torch
from google.colab import files

PRAGMA_HEADLESS = os.environ.get("PRAGMA_HEADLESS") == "1"   # 1 = Colab CLI, sin navegador

def select_precision(cuda_available, capability=None):
    if not cuda_available:
        return {"device": "cpu", "dtype": "float32", "autocast": False}
    native_bf16 = int(capability[0]) >= 8
    return {"device": "cuda", "dtype": "bfloat16" if native_bf16 else "float16", "autocast": True}

CUDA_AVAILABLE = torch.cuda.is_available()
CUDA_CAPABILITY = torch.cuda.get_device_capability(0) if CUDA_AVAILABLE else None
PRECISION = select_precision(CUDA_AVAILABLE, CUDA_CAPABILITY)
DEVICE = PRECISION["device"]
TORCH_DTYPE = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[PRECISION["dtype"]]

def inference_precision():
    return torch.autocast(device_type="cuda", dtype=TORCH_DTYPE) if DEVICE == "cuda" else nullcontext()

def synchronize():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
    DEVICE_NAME = torch.cuda.get_device_name(0)
else:
    DEVICE_NAME = "CPU"
    print("ADVERTENCIA: sin GPU. La corrida se registrará como REAL_CPU y no será comparable.")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = WORK_DIR / "runs_v14" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

def sha256_file(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

def find_acceptance_image(folders):
    for folder in map(Path, folders):
        if not folder.is_dir():
            continue
        for path in sorted(folder.iterdir()):
            if (path.is_file() and path.suffix.lower() in (".jpg", ".jpeg")
                    and path.stat().st_size == EXPECTED_IMAGE_BYTES and sha256_file(path) == EXPECTED_IMAGE_SHA256):
                return path
    return None

IMAGE_PATH = find_acceptance_image([Path("/content"), Path("/mnt/data"), WORK_DIR])
if IMAGE_PATH is not None:
    print(f"Foto encontrada por su huella SHA-256: {IMAGE_PATH}")
elif PRAGMA_HEADLESS:
    raise FileNotFoundError("Modo sin navegador: sube antes la foto a /content.")
else:
    print("Selecciona la foto P1070614.JPG. Se validará por SHA-256; el nombre puede variar.")
    uploaded = files.upload()
    exact = [(n, p) for n, p in uploaded.items() if hashlib.sha256(p).hexdigest() == EXPECTED_IMAGE_SHA256]
    if len(exact) != 1:
        raise ValueError("No se recibió exactamente la fotografía de aceptación.")
    IMAGE_PATH = RUN_DIR / "P1070614.JPG"
    IMAGE_PATH.write_bytes(exact[0][1])

IMAGE_SHA256 = sha256_file(IMAGE_PATH)
assert IMAGE_SHA256 == EXPECTED_IMAGE_SHA256, "La foto no coincide con la aceptación acordada."
image = np.asarray(ImageOps.exif_transpose(Image.open(IMAGE_PATH)).convert("RGB"))
assert image.shape[:2] == (2248, 4000), image.shape
ENVIRONMENT = {
    "python": platform.python_version(), "torch": torch.__version__, "device": DEVICE, "device_name": DEVICE_NAME,
    "cuda_capability": CUDA_CAPABILITY, "dtype": PRECISION["dtype"], "sam2_commit": SAM2_COMMIT,
    "checkpoint_bytes": CHECKPOINT.stat().st_size, "checkpoint_sha256": sha256_file(CHECKPOINT),
    "image_sha256": IMAGE_SHA256, "image_size": [image.shape[1], image.shape[0]], "run_id": RUN_ID,
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))

In [ ]:
# Celda 3 · prerregistro embebido (no editar) y compuerta de luma de TODOS los prompts
PREREG_FILE_SHA256 = "b0cfe33ee6aca8921c3e0c7a01a2604131b5e6cf9a448d6b8635626f13c690aa"
PREREG_TEXT = "{\n  \"schema\": \"pragma.aem1_preregistration\",\n  \"schema_version\": \"0.1.0\",\n  \"experiment\": \"A-E(−1) v1.4 · caso chica · una sola intervención (H2)\",\n  \"status\": \"PREREGISTERED\",\n  \"frozen_on\": \"2026-09-26\",\n  \"responds_to\": [\n    \"dialogo/005_chatgpt_a_claude.md\",\n    \"dialogo/006_chatgpt_a_claude.md\"\n  ],\n  \"go_to_preregistration\": {\n    \"by\": \"ChatGPT\",\n    \"letter\": \"dialogo/005_chatgpt_a_claude.md\",\n    \"letter_sha256\": \"dd8aa20fb0efdf05fe71c6826dde952c3aab8a681ca3d6783518db7206376d60\",\n    \"accepted\": [\n      \"una sola intervención nueva\",\n      \"cadena +POS_HAIR+SLEEVE · box+corrections\",\n      \"tres semillas\",\n      \"referencia v1.3 obligatoriamente bit a bit\",\n      \"H2 únicamente como positivo adicional\",\n      \"ningún otro cambio\",\n      \"H-C3 congelada tal cual\",\n      \"H-G5 añadida antes de los datos\",\n      \"PASS = contrato completo, no «H2 cerró el agujero»\"\n    ],\n    \"cross_audit_before_run\": \"ver cross_audit (ChatGPT 006)\"\n  },\n  \"cross_audit\": {\n    \"by\": \"ChatGPT\",\n    \"letter\": \"dialogo/006_chatgpt_a_claude.md\",\n    \"letter_sha256\": \"9587f1c0b04a6dd852b96a9f014603ba183727d7cb9504f8b4d27534e7309ddf\",\n    \"inspected\": {\n      \"package_sha256\": \"635640565d3e3efbbad6ce3f1c6d30cdcaed1e45dbefb46676f3d9ae516adc5e\",\n      \"prereg_content_sha256\": \"bd437a87a33651ab7103f6934ea7fd6934b16c1cc29cc069ed376293dafc15a3\"\n    },\n    \"verdicts\": {\n      \"ZIP_INTEGRITY\": \"PASS\",\n      \"H2_OWNER_SECOND_KEY\": \"PASS\",\n      \"H2_VALID_PERTURBATIONS\": \"5/5 PASS\",\n      \"SIX_SHEET_BLIND_PACKAGE\": \"ACCEPTED\",\n      \"STOP_RULE\": \"CONFIRMED\",\n      \"DEC_024\": \"ACCEPTED\",\n      \"H-G5\": \"CHANGE_REQUIRED (solo la definición de new_d)\",\n      \"AEM1_v1.4\": \"HOLD_FOR_ONE_PREREG_PATCH\"\n    },\n    \"changes_applied_before_any_run\": [\n      \"new_d: «nuevo fuera de H2» = agujero ∩ referencia ∩ ¬región ≥ 1000 px (ChatGPT 006 §2). Antes: el agujero se descartaba si tocaba la región en 1 px (caso A) y se exigía ≥ 50 % de su área dentro de la referencia (caso B); la fracción queda solo como diagnóstico\",\n      \"cuatro pruebas sintéticas de ChatGPT 006 añadidas a tests/test_aem1_v14.py antes de regenerar\"\n    ],\n    \"unchanged\": \"H2, ramas, hipótesis (salvo new_d), seis láminas, PASS y regla de parada\",\n    \"result\": \"pendiente de que ChatGPT confirme este delta; entonces AEM1_v1.4 = GO_TO_GPU\"\n  },\n  \"correction_of_letter_005\": {\n    \"proposed\": [\n      2964,\n      672\n    ],\n    \"problem\": \"(2964, 672) está en la parte ALTA de la franja: agujero D solo en s0 y s2; C01 (s1) ya cubre esa parte y su único agujero D es la parte BAJA (2984–3025 × 842–997). Con ese punto la hipótesis no podía probarse en el mejor intento\",\n    \"fix\": \"H2 se elige dentro de los píxeles que son agujero D de consenso en las tres semillas (la parte baja)\",\n    \"second_key_on_old_point\": \"ChatGPT 005 confirmó el punto viejo (pelo de la chica); ChatGPT 006 confirmó el nuevo y sus 5 perturbaciones válidas\",\n    \"upper_probe_role\": \"solo descriptivo: si la parte alta de s0 y s2 también se cierra\"\n  },\n  \"generated_by\": \"work/design_aem1_v1_4.py\",\n  \"image\": {\n    \"sha256\": \"8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d\",\n    \"size\": [\n      4000,\n      2248\n    ],\n    \"orientation\": \"EXIF aplicado; sin redimensionar\"\n  },\n  \"sam2_freeze\": {\n    \"SAM2_GIT_COMMIT\": \"2b90b9f5ceec907a1c18123530e92e794ad901a4\",\n    \"MODEL_CONFIG\": \"configs/sam2.1/sam2.1_hiera_l.yaml\",\n    \"CHECKPOINT\": \"sam2.1_hiera_large.pt\",\n    \"CHECKPOINT_SHA256\": \"2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318\",\n    \"CHECKPOINT_BYTES\": 898083611,\n    \"dtype\": \"bfloat16 (autocast CUDA), como la corrida 1\",\n    \"gate\": \"el cuaderno bloquea antes de generar si el commit, el checkpoint o la foto difieren\",\n    \"reproducibility\": \"GPU NVIDIA L4, como la corrida 1 y v1.3 (la reproducción bit a bit se probó en L4)\"\n  },\n  \"base_config\": {\n    \"source_notebook\": \"outputs/PRAGMA_A-E-menos-1_diagnostico_caso_chica_v1_2.ipynb\",\n    \"source_notebook_sha256\": \"06315e0fe15b84b446570903d7a6c5c219df5ea2ecf4577429cf107cdcdef0da\",\n    \"run1_config_digest\": \"fd29b18bbe50cf8236f41057567265e6083d13465511e4882cd8e533af58aa71\",\n    \"BOX\": [\n      2100,\n      300,\n      3500,\n      2247\n    ],\n    \"prompts\": [\n      {\n        \"id\": \"P+1\",\n        \"xy\": [\n          2588,\n          1785\n        ],\n        \"description\": \"botón/overol en el torso de la chica\",\n        \"patch_luma_mean\": 249.3,\n        \"patch_luma_std\": 6.2,\n        \"description_v1_2\": \"torso de la chica\"\n      },\n      {\n        \"id\": \"P-1\",\n        \"xy\": [\n          2350,\n          900\n        ],\n        \"description\": \"torso floral posterior\",\n        \"patch_luma_mean\": 125.6,\n        \"patch_luma_std\": 71.0\n      },\n      {\n        \"id\": \"P-2\",\n        \"xy\": [\n          2640,\n          430\n        ],\n        \"description\": \"cabello recogido posterior (v1.1: antes borde con pared)\",\n        \"patch_luma_mean\": 56.8,\n        \"patch_luma_std\": 44.9\n      },\n      {\n        \"id\": \"P-3\",\n        \"xy\": [\n          2400,\n          810\n        ],\n        \"description\": \"hombro posterior, tela oscura (v1.1: antes pared)\",\n        \"patch_luma_mean\": 90.9,\n        \"patch_luma_std\": 53.0\n      }\n    ],\n    \"holdouts\": {\n      \"keep_subject\": [\n        {\n          \"id\": \"K1\",\n          \"xy\": [\n            2835,\n            635\n          ],\n          \"description\": \"cara\",\n          \"patch_luma_mean\": 245.8,\n          \"patch_luma_std\": 2.9\n        },\n        {\n          \"id\": \"K2\",\n          \"xy\": [\n            2730,\n            650\n          ],\n          \"description\": \"cabello frontal\",\n          \"patch_luma_mean\": 135.3,\n          \"patch_luma_std\": 51.7\n        },\n        {\n          \"id\": \"K3\",\n          \"xy\": [\n            2435,\n            1135\n          ],\n          \"description\": \"hombro/ropa izquierda\",\n          \"patch_luma_mean\": 101.4,\n          \"patch_luma_std\": 57.1\n        },\n        {\n          \"id\": \"K4\",\n          \"xy\": [\n            3185,\n            985\n          ],\n          \"description\": \"mano levantada\",\n          \"patch_luma_mean\": 254.3,\n          \"patch_luma_std\": 0.8\n        },\n        {\n          \"id\": \"K5\",\n          \"xy\": [\n            3285,\n            1335\n          ],\n          \"description\": \"manga/brazo derecho\",\n          \"patch_luma_mean\": 101.7,\n          \"patch_luma_std\": 53.1\n        },\n        {\n          \"id\": \"K6\",\n          \"xy\": [\n            2235,\n            1935\n          ],\n          \"description\": \"mano que cuelga\",\n          \"patch_luma_mean\": 220.3,\n          \"patch_luma_std\": 23.7\n        },\n        {\n          \"id\": \"K7\",\n          \"xy\": [\n            2785,\n            2035\n          ],\n          \"description\": \"torso inferior\",\n          \"patch_luma_mean\": 252.9,\n          \"patch_luma_std\": 1.7\n        }\n      ],\n      \"drop_other_person\": [\n        {\n          \"id\": \"O1\",\n          \"xy\": [\n            2460,\n            860\n          ],\n          \"description\": \"ropa floral posterior\",\n          \"patch_luma_mean\": 179.8,\n          \"patch_luma_std\": 57.1\n        },\n        {\n          \"id\": \"O2\",\n          \"xy\": [\n            2700,\n            380\n          ],\n          \"description\": \"cabello recogido posterior (v1.1: antes pared)\",\n          \"patch_luma_mean\": 78.7,\n          \"patch_luma_std\": 52.6\n        },\n        {\n          \"id\": \"O3\",\n          \"xy\": [\n            2735,\n            360\n          ],\n          \"description\": \"cabello recogido posterior, arriba (v1.2: antes zona de contacto)\",\n          \"patch_luma_mean\": 57.1,\n          \"patch_luma_std\": 43.5\n        },\n        {\n          \"id\": \"O4\",\n          \"xy\": [\n            2325,\n            965\n          ],\n          \"description\": \"blusa floral posterior, interior (v1.2: antes borde)\",\n          \"patch_luma_mean\": 185.1,\n          \"patch_luma_std\": 36.5\n        }\n      ],\n      \"drop_background\": [\n        {\n          \"id\": \"B1\",\n          \"xy\": [\n            2185,\n            435\n          ],\n          \"description\": \"cuadro\",\n          \"patch_luma_mean\": 203.4,\n          \"patch_luma_std\": 28.8\n        },\n        {\n          \"id\": \"B2\",\n          \"xy\": [\n            1735,\n            1635\n          ],\n          \"description\": \"mesa/mantel\",\n          \"patch_luma_mean\": 182.9,\n          \"patch_luma_std\": 35.6\n        },\n        {\n          \"id\": \"B3\",\n          \"xy\": [\n            3485,\n            1485\n          ],\n          \"description\": \"sofá/fondo\",\n          \"patch_luma_mean\": 49.6,\n          \"patch_luma_std\": 39.4\n        }\n      ]\n    },\n    \"patch_radius\": 6,\n    \"keep_min_coverage\": 0.8,\n    \"drop_max_coverage\": 0.2\n  },\n  \"new_prompts\": {\n    \"H1\": {\n      \"xy\": [\n        3072,\n        592\n      ],\n      \"owner_expected\": \"chica\",\n      \"material\": \"pelo oscuro\",\n      \"declared_window_xyxy\": [\n        2940,\n        400,\n        3200,\n        900\n      ],\n      \"window_rationale\": \"pelo de la chica a la derecha de su cabeza (lado opuesto al moño), fuera de la caja de contacto\",\n      \"selection_rule\": \"pragma_ae.aem1_v13.select_safe_point (rejilla de 2 px)\",\n      \"safe_square_half_px\": 30,\n      \"safe_square_dark_fraction\": 0.9817,\n      \"contact_box_margin_linf_px\": 173,\n      \"frontier_distance_lower_bound_px\": 173,\n      \"nearest_holdout\": {\n        \"id\": \"K1\",\n        \"distance_px\": 240.9\n      },\n      \"nearest_prompt\": {\n        \"id\": \"P-2\",\n        \"distance_px\": 461.4\n      },\n      \"patch_luma_mean\": 70.8,\n      \"patch_luma_std\": 48.4,\n      \"image_sha256\": \"8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d\",\n      \"owner_verified_by\": \"auditor IA Claude en la lámina privada de diseño\",\n      \"owner_second_key\": \"PASS (ChatGPT 003: el punto y sus 8 perturbaciones sobre la chica)\"\n    },\n    \"S1\": {\n      \"xy\": [\n        2230,\n        1686\n      ],\n      \"owner_expected\": \"chica\",\n      \"material\": \"manga oscura\",\n      \"declared_window_xyxy\": [\n        2120,\n        1350,\n        2340,\n        1850\n      ],\n      \"window_rationale\": \"manga oscura del brazo que cuelga, por debajo de la caja de contacto\",\n      \"selection_rule\": \"pragma_ae.aem1_v13.select_safe_point (rejilla de 2 px)\",\n      \"safe_square_half_px\": 85,\n      \"safe_square_dark_fraction\": 0.9842,\n      \"contact_box_margin_linf_px\": 387,\n      \"frontier_distance_lower_bound_px\": 387,\n      \"nearest_holdout\": {\n        \"id\": \"K6\",\n        \"distance_px\": 249.1\n      },\n      \"nearest_prompt\": {\n        \"id\": \"P+1\",\n        \"distance_px\": 371.4\n      },\n      \"patch_luma_mean\": 66.9,\n      \"patch_luma_std\": 49.4,\n      \"image_sha256\": \"8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d\",\n      \"owner_verified_by\": \"auditor IA Claude en la lámina privada de diseño\",\n      \"owner_second_key\": \"PASS (ChatGPT 003: el punto y sus 8 perturbaciones sobre la chica)\"\n    },\n    \"H2\": {\n      \"xy\": [\n        2994,\n        892\n      ],\n      \"owner_expected\": \"chica\",\n      \"material\": \"pelo oscuro visible entre la cara y el índice levantado (parte baja de la franja)\",\n      \"region_rule\": \"píxeles que las dos llaves de v1.3 marcaron como agujero D (consenso) en las TRES semillas de referencia +POS_HAIR+SLEEVE|box+corrections\",\n      \"region\": {\n        \"px\": 2973,\n        \"bbox_xyxy\": [\n          2984,\n          842,\n          3025,\n          997\n        ],\n        \"packed_sha256\": \"fb60cd3e6cc5835d7aa537151f1a153ab61ea8083b6b7887e2dbe20f7e88cc56\"\n      },\n      \"selection_rule\": \"pragma_ae.aem1_v14.select_point_in_region: rejilla par; mayor cuadrado seguro ≥ 98 % oscuro; empates por fracción oscura, margen a la caja de contacto, menor y, menor x\",\n      \"safe_square_half_min_px\": 7,\n      \"safe_square_half_px\": 15,\n      \"safe_square_dark_fraction\": 0.990635,\n      \"why_smaller_than_H1_S1\": \"la franja mide ~40 px de ancho: el mayor cuadrado seguro dentro de la región es de 31 px, por debajo del mínimo de 43 px de H1/S1. Consecuencia declarada: las perturbaciones que sacan el parche del pelo son INVALID_PERTURBATION y no se ejecutan\",\n      \"contact_box_margin_linf_px\": 95,\n      \"nearest_holdout\": {\n        \"id\": \"K4\",\n        \"distance_px\": 212.4\n      },\n      \"nearest_prompt\": {\n        \"id\": \"H1\",\n        \"distance_px\": 310.0\n      },\n      \"patch_luma_mean\": 97.8,\n      \"patch_luma_std\": 54.7,\n      \"image_sha256\": \"8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d\",\n      \"inside_target_hole_of_each_reference_seed\": {\n        \"s0\": {\n          \"target_hole\": {\n            \"number\": 2,\n            \"area\": 7575,\n            \"bbox\": [\n              2969,\n              789,\n              3027,\n              1000\n            ]\n          },\n          \"upper_probe_hole\": {\n            \"number\": 3,\n            \"area\": 4176,\n            \"bbox\": [\n              2950,\n              636,\n              2999,\n              779\n            ]\n          }\n        },\n        \"s1\": {\n          \"target_hole\": {\n            \"number\": 2,\n            \"area\": 3011,\n            \"bbox\": [\n              2984,\n              842,\n              3025,\n              997\n            ]\n          },\n          \"upper_probe_hole\": null\n        },\n        \"s2\": {\n          \"target_hole\": {\n            \"number\": 3,\n            \"area\": 8120,\n            \"bbox\": [\n              2964,\n              785,\n              3026,\n              999\n            ]\n          },\n          \"upper_probe_hole\": {\n            \"number\": 4,\n            \"area\": 5027,\n            \"bbox\": [\n              2949,\n              636,\n              2999,\n              781\n            ]\n          }\n        }\n      },\n      \"owner_verified_by\": \"auditor IA Claude en la lámina privada de diseño\",\n      \"owner_second_key\": \"PASS (ChatGPT 006: H2 y sus 5 perturbaciones válidas sobre el pelo de la chica; las 3 hacia +x, bien marcadas inválidas)\"\n    }\n  },\n  \"safety_constants\": {\n    \"contact_box_xyxy\": [\n      2150,\n      250,\n      2900,\n      1300\n    ],\n    \"contact_box_claim\": \"toda la frontera visible chica↔persona posterior está dentro de esta caja; su margen es cota inferior de la distancia a la frontera\",\n    \"contact_margin_min_linf_px\": 40,\n    \"perturbed_contact_margin_min_linf_px\": 25,\n    \"holdout_distance_min_px\": 100,\n    \"perturbed_holdout_distance_min_px\": 78,\n    \"prompt_distance_min_px\": 100,\n    \"dark_luma_threshold\": 110,\n    \"dark_blur_radius\": 4,\n    \"safe_square_dark_fraction_min\": 0.98,\n    \"safe_square_half_min_px\": 21,\n    \"perturbed_patch_dark_fraction_min\": 0.9,\n    \"luma\": \"media de los canales RGB en el parche 13×13, como el preflight v1.2\",\n    \"runtime_binding\": \"el cuaderno recalcula la luma de TODO prompt (base y perturbado) y bloquea si difiere > 3,0 del valor registrado aquí\"\n  },\n  \"invalid_perturbation\": \"un caso INVALID_PERTURBATION no se ejecuta y nunca cuenta como evidencia contra SAM 2\",\n  \"reference\": {\n    \"v1_3_run\": \"20260926T040705Z_bee282c1\",\n    \"v1_3_zip_sha256\": \"6d795132cd8e609388cdb1b3671dee7e0d1c39067880cfb0ca6a1b1a6783ce14\",\n    \"v1_3_prereg_content_sha256\": \"5800f2bf624620c353067210c73d782a8aefa39563925e859929ef0cba432a8c\",\n    \"v1_3_closure\": {\n      \"file\": \"auditoria/aem1v13_20260926T040705Z_bee282c1/cierre_chatgpt005.json\",\n      \"sha256\": \"1922d2a66cec32786b71350d25cec924e3880323f5530c44f5faf006160abcbe\",\n      \"AEM1_v1_3\": \"CLOSED_INCONCLUSIVE\"\n    },\n    \"chain\": \"+POS_HAIR+SLEEVE|box+corrections\",\n    \"v1_3_packed_mask_sha256\": {\n      \"BASE|box|0\": \"2e3cc1c8ef21fa99919506b1e2fbb4d8e24e38ac93fce7e31e272fb2e05590aa\",\n      \"BASE|box|1\": \"9aa2c41783b0593ca8d702104b1d8756b9c0c08450c1bf9b0e9afdff2ac624e8\",\n      \"BASE|box|2\": \"7fa08b3ddb75b6e585811caea4d65ffab19f38a621af64cdf2ade28cce011466\",\n      \"+POS_HAIR+SLEEVE|box+corrections|s0\": \"179502200aa2829aed1672886434a4ad8c464304bf8ac6f9f678540d5ee09243\",\n      \"+POS_HAIR+SLEEVE|box+corrections|s1\": \"2815486d00ea3bc9bae00834b8dd1b04f4a350a243efd3b2f8986661a8bd070a\",\n      \"+POS_HAIR+SLEEVE|box+corrections|s2\": \"0423308b83ed6043c438e13c96fb9b0bea90c0c748e15f2bb49cde969db8bbfe\"\n    },\n    \"bit_exact_rule\": \"BASE|box y las 3 semillas de referencia deben salir bit a bit iguales a v1.3. Si alguna no lo es, la comparación de esa semilla usa la referencia de esta corrida y su consenso ciego\",\n    \"judgments\": {\n      \"s0\": {\n        \"candidate_id\": \"+POS_HAIR+SLEEVE|box+corrections|s0\",\n        \"v1_3_label\": \"C03\",\n        \"correct_subject\": \"TRUE\",\n        \"body_and_edges_complete\": \"FALSE\",\n        \"other_person_excluded\": \"FALSE\",\n        \"background_excluded\": \"TRUE\",\n        \"d_holes\": [\n          \"2\",\n          \"3\",\n          \"4\"\n        ],\n        \"holes\": [\n          {\n            \"number\": 1,\n            \"area\": 11120,\n            \"bbox\": [\n              2208,\n              1962,\n              2284,\n              2177\n            ],\n            \"consensus\": \"L\"\n          },\n          {\n            \"number\": 2,\n            \"area\": 7575,\n            \"bbox\": [\n              2969,\n              789,\n              3027,\n              1000\n            ],\n            \"consensus\": \"D\"\n          },\n          {\n            \"number\": 3,\n            \"area\": 4176,\n            \"bbox\": [\n              2950,\n              636,\n              2999,\n              779\n            ],\n            \"consensus\": \"D\"\n          },\n          {\n            \"number\": 4,\n            \"area\": 2985,\n            \"bbox\": [\n              2683,\n              832,\n              2730,\n              930\n            ],\n            \"consensus\": \"D\"\n          }\n        ],\n        \"o_core_px\": 5,\n        \"keys_v1_3\": {\n          \"claude\": {\n            \"correct_subject\": \"TRUE\",\n            \"body_and_edges_complete\": \"FALSE\",\n            \"other_person_excluded\": \"TRUE\",\n            \"background_excluded\": \"TRUE\"\n          },\n          \"chatgpt\": {\n            \"correct_subject\": \"TRUE\",\n            \"body_and_edges_complete\": \"FALSE\",\n            \"other_person_excluded\": \"FALSE\",\n            \"background_excluded\": \"TRUE\"\n          }\n        }\n      },\n      \"s1\": {\n        \"candidate_id\": \"+POS_HAIR+SLEEVE|box+corrections|s1\",\n        \"v1_3_label\": \"C01\",\n        \"correct_subject\": \"TRUE\",\n        \"body_and_edges_complete\": \"FALSE\",\n        \"other_person_excluded\": \"TRUE\",\n        \"background_excluded\": \"TRUE\",\n        \"d_holes\": [\n          \"2\"\n        ],\n        \"holes\": [\n          {\n            \"number\": 1,\n            \"area\": 10903,\n            \"bbox\": [\n              2209,\n              1965,\n              2287,\n              2175\n            ],\n            \"consensus\": \"L\"\n          },\n          {\n            \"number\": 2,\n            \"area\": 3011,\n            \"bbox\": [\n              2984,\n              842,\n              3025,\n              997\n            ],\n            \"consensus\": \"D\"\n          }\n        ],\n        \"o_core_px\": 0,\n        \"keys_v1_3\": {\n          \"claude\": {\n            \"correct_subject\": \"TRUE\",\n            \"body_and_edges_complete\": \"FALSE\",\n            \"other_person_excluded\": \"TRUE\",\n            \"background_excluded\": \"TRUE\"\n          },\n          \"chatgpt\": {\n            \"correct_subject\": \"TRUE\",\n            \"body_and_edges_complete\": \"FALSE\",\n            \"other_person_excluded\": \"TRUE\",\n            \"background_excluded\": \"TRUE\"\n          }\n        }\n      },\n      \"s2\": {\n        \"candidate_id\": \"+POS_HAIR+SLEEVE|box+corrections|s2\",\n        \"v1_3_label\": \"C10\",\n        \"correct_subject\": \"TRUE\",\n        \"body_and_edges_complete\": \"FALSE\",\n        \"other_person_excluded\": \"TRUE\",\n        \"background_excluded\": \"TRUE\",\n        \"d_holes\": [\n          \"2\",\n          \"3\",\n          \"4\"\n        ],\n        \"holes\": [\n          {\n            \"number\": 1,\n            \"area\": 11506,\n            \"bbox\": [\n              2209,\n              1963,\n              2285,\n              2177\n            ],\n            \"consensus\": \"L\"\n          },\n          {\n            \"number\": 2,\n            \"area\": 10793,\n            \"bbox\": [\n              2651,\n              830,\n              2822,\n              1021\n            ],\n            \"consensus\": \"D\"\n          },\n          {\n            \"number\": 3,\n            \"area\": 8120,\n            \"bbox\": [\n              2964,\n              785,\n              3026,\n              999\n            ],\n            \"consensus\": \"D\"\n          },\n          {\n            \"number\": 4,\n            \"area\": 5027,\n            \"bbox\": [\n              2949,\n              636,\n              2999,\n              781\n            ],\n            \"consensus\": \"D\"\n          }\n        ],\n        \"o_core_px\": 0,\n        \"keys_v1_3\": {\n          \"claude\": {\n            \"correct_subject\": \"TRUE\",\n            \"body_and_edges_complete\": \"FALSE\",\n            \"other_person_excluded\": \"TRUE\",\n            \"background_excluded\": \"TRUE\"\n          },\n          \"chatgpt\": {\n            \"correct_subject\": \"TRUE\",\n            \"body_and_edges_complete\": \"FALSE\",\n            \"other_person_excluded\": \"TRUE\",\n            \"background_excluded\": \"TRUE\"\n          }\n        }\n      }\n    }\n  },\n  \"branches\": {\n    \"BASE\": {\n      \"purpose\": \"semillas: la misma llamada box (multimask) de v1.3\",\n      \"protocols\": {\n        \"box\": \"BOX, multimask, 3 salidas\"\n      }\n    },\n    \"+POS_HAIR+SLEEVE\": {\n      \"purpose\": \"referencia: la cadena ganadora de v1.3, idéntica llamada a llamada\",\n      \"points\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ]\n    },\n    \"+POS_HAIR+SLEEVE+H2\": {\n      \"purpose\": \"la intervención: H2 como cuarto positivo en la corrección; nada más cambia\",\n      \"points\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ]\n    },\n    \"PERTURB_POINT\": {\n      \"offsets_px\": [\n        [\n          15,\n          0\n        ],\n        [\n          -15,\n          0\n        ],\n        [\n          0,\n          15\n        ],\n        [\n          0,\n          -15\n        ],\n        [\n          15,\n          15\n        ],\n        [\n          15,\n          -15\n        ],\n        [\n          -15,\n          15\n        ],\n        [\n          -15,\n          -15\n        ]\n      ],\n      \"linf_radius_px\": 15,\n      \"one_at_a_time\": true,\n      \"targets\": {\n        \"H2\": {\n          \"base\": \"H2\",\n          \"base_xy\": [\n            2994,\n            892\n          ],\n          \"branch\": \"+POS_HAIR+SLEEVE+H2\",\n          \"protocols\": [\n            \"box+corrections\"\n          ],\n          \"perturbations\": [\n            {\n              \"id\": \"d+15+0\",\n              \"dx\": 15,\n              \"dy\": 0,\n              \"xy\": [\n                3009,\n                892\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": false,\n              \"reasons\": [\n                \"INVALID_PERTURBATION: el parche deja el material oscuro (0.56 < 0.9)\"\n              ],\n              \"patch_luma_mean\": 117.0,\n              \"patch_luma_std\": 69.4\n            },\n            {\n              \"id\": \"d-15+0\",\n              \"dx\": -15,\n              \"dy\": 0,\n              \"xy\": [\n                2979,\n                892\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 93.3,\n              \"patch_luma_std\": 53.5\n            },\n            {\n              \"id\": \"d+0+15\",\n              \"dx\": 0,\n              \"dy\": 15,\n              \"xy\": [\n                2994,\n                907\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 101.2,\n              \"patch_luma_std\": 50.3\n            },\n            {\n              \"id\": \"d+0-15\",\n              \"dx\": 0,\n              \"dy\": -15,\n              \"xy\": [\n                2994,\n                877\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 95.5,\n              \"patch_luma_std\": 54.5\n            },\n            {\n              \"id\": \"d+15+15\",\n              \"dx\": 15,\n              \"dy\": 15,\n              \"xy\": [\n                3009,\n                907\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": false,\n              \"reasons\": [\n                \"INVALID_PERTURBATION: el parche deja el material oscuro (0.60 < 0.9)\"\n              ],\n              \"patch_luma_mean\": 105.3,\n              \"patch_luma_std\": 57.7\n            },\n            {\n              \"id\": \"d+15-15\",\n              \"dx\": 15,\n              \"dy\": -15,\n              \"xy\": [\n                3009,\n                877\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": false,\n              \"reasons\": [\n                \"INVALID_PERTURBATION: el parche deja el material oscuro (0.42 < 0.9)\"\n              ],\n              \"patch_luma_mean\": 135.0,\n              \"patch_luma_std\": 81.5\n            },\n            {\n              \"id\": \"d-15+15\",\n              \"dx\": -15,\n              \"dy\": 15,\n              \"xy\": [\n                2979,\n                907\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 99.5,\n              \"patch_luma_std\": 54.2\n            },\n            {\n              \"id\": \"d-15-15\",\n              \"dx\": -15,\n              \"dy\": -15,\n              \"xy\": [\n                2979,\n                877\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 88.3,\n              \"patch_luma_std\": 57.5\n            }\n          ],\n          \"valid_count\": 5,\n          \"chain\": \"semillas de BASE sin cambios; solo H2 se perturba (H1 y S1 fijos)\"\n        }\n      }\n    }\n  },\n  \"combination\": \"NINGUNA\",\n  \"candidate_counts\": {\n    \"BASE|box\": 3,\n    \"REF\": 3,\n    \"H2\": 3,\n    \"PERTURB_POINT\": 15,\n    \"valid_point_perturbations\": 5,\n    \"total_masks\": 24,\n    \"calls\": 22\n  },\n  \"call_plan\": [\n    {\n      \"call_id\": \"BASE|box\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"box\",\n      \"point_ids\": [],\n      \"points\": [],\n      \"labels\": [],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"BASE|box|0\",\n        \"BASE|box|1\",\n        \"BASE|box|2\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|box+corrections|s0\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|box+corrections|s1\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|box+corrections|s2\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE+H2|box+corrections|s0\",\n      \"branch\": \"+POS_HAIR+SLEEVE+H2\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          892\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE+H2|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE+H2|box+corrections|s1\",\n      \"branch\": \"+POS_HAIR+SLEEVE+H2\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          892\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE+H2|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE+H2|box+corrections|s2\",\n      \"branch\": \"+POS_HAIR+SLEEVE+H2\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          892\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE+H2|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15+0|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          892\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15+0|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15+0|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          892\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15+0|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15+0|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          892\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15+0|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d+0+15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          907\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d+0+15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d+0+15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          907\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d+0+15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d+0+15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          907\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d+0+15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d+0-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          877\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d+0-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d+0-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          877\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d+0-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d+0-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2994,\n          877\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d+0-15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15+15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          907\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15+15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15+15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          907\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15+15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15+15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          907\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15+15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          877\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          877\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H2|d-15-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H2\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"H2\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2979,\n          877\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H2|d-15-15|box+corrections|s2\"\n      ]\n    }\n  ],\n  \"call_plan_semantics\": \"cada llamada es SAM2ImagePredictor.predict(point_coords=points o None, point_labels=labels o None, box=box o None, mask_input=low_res_logits[index][None] de mask_input_from o None, multimask_output, return_logits=True); máscara = logits > 0; mismas conversiones que v1.2 (float32 para puntos y caja, int32 para etiquetas)\",\n  \"analysis_implementation_sha256\": {\n    \"pragma_ae/aem1_v14.py\": \"ee6cdecd07a7d79adee94109c93ea4a7b76ab55da291d18343f6816bc1c7342d\",\n    \"pragma_ae/aem1_v14_audit.py\": \"595b5f42d041864f9a614c7519829bacbee7b7961ee6694af6d7a3c6e86e4a44\",\n    \"pragma_ae/aem1_v13.py\": \"2e9433604a834db8b5636d112dcd5169b5330727e0a0c05c4e92e9cbc4087da7\",\n    \"pragma_ae/aem1_v13_audit.py\": \"820af07186a171099f94e98541d89d0b0948a3cf4333222c4c0c5a77ebfdc13a\",\n    \"pragma_ae/masks.py\": \"412d1ead02fb92891ca989c5c6ac7a5b2dcd5b6bcdbb57551ec1ce12db712ac9\"\n  },\n  \"blind_audit\": {\n    \"protocol\": \"auditoria/PROTOCOLO_AUDITORIA_AEM1_v2.md\",\n    \"protocol_sha256\": \"b5b11c6cce30b534eb83ab9117025e35837e4554eb150e411c8091fec5087a71\",\n    \"candidates\": \"6 láminas N01–N06: las 3 semillas con H2 y las 3 de referencia, mezcladas al azar\",\n    \"why_reference_is_included\": \"señuelo y retest: la persona que juzga no sabe cuáles tienen H2, y se mide si cada llave repite su juicio de v1.3 sobre la misma máscara\",\n    \"reference_judgment_if_bit_exact\": \"la doble llave de v1.3 (congelada arriba); el retest solo describe\",\n    \"not_blind_judged\": [\n      \"BASE|box\",\n      \"PERTURB_POINT\"\n    ],\n    \"order\": \"el paquete va a ChatGPT antes que cualquier resultado; los juicios de Claude se comprometen por hash\"\n  },\n  \"adjudication_rules\": {\n    \"other_person_excluded\": \"discrepancia → medición prerregistrada: FALSE si hay píxeles de máscara en el núcleo oscuro del moño [2610, 315, 2780, 415] o islas separadas en el núcleo hombro/blusa [2400, 840, 2520, 990]; si una llave cita material posterior fuera de los núcleos, con coordenadas, tercera revisión ciega (§5.2)\",\n    \"aux\": \"discrepancia → NO_CONSENSO (las auxiliares no deciden nada en v1.4)\",\n    \"correct_subject, body_and_edges_complete, background_excluded\": \"protocolo v2 rev. 1 §5.2\",\n    \"hole_D_L\": \"solo se adjudica si el agujero cuenta para H-G5 (agujero nuevo fuera de la región de H2); §5.2\"\n  },\n  \"metrics\": {\n    \"target_hole\": \"por semilla: el agujero cerrado ≥ 1000 px de la referencia que contiene H2; es evaluable si las dos llaves lo marcaron D\",\n    \"closure\": \"cerrado = quedan < 1000 px del agujero objetivo sin cubrir Y ningún agujero ≥ 1000 px de la candidata lo toca (pragma_ae.aem1_v14.closure)\",\n    \"h2_region\": \"el agujero objetivo dilatado 15 px (L∞)\",\n    \"new_d_hole\": \"agujero ≥ 1000 px de la candidata cuya pérdida nueva fuera de la región de H2 (agujero ∩ máscara de referencia ∩ ¬región) es ≥ 1000 px, toque o no la región, y D en las dos llaves (ChatGPT 006). La fracción del agujero que ya estaba dentro de la referencia solo se reporta\",\n    \"o_worse\": \"referencia O TRUE y candidata O no TRUE; o, si la referencia ya era FALSE (s0: isla de 5 px en el moño), más píxeles en los núcleos que la referencia\",\n    \"perturbation_stability\": \"pragma_ae.aem1_v13.perturbation_stability por semilla, más cierre bajo cada perturbación\",\n    \"descriptive_only\": [\n      \"cambio global (IoU global y fuera de la región, píxeles perdidos y ganados fuera)\",\n      \"cierre de la parte alta de la franja en s0 y s2 (sonda (2964, 672))\",\n      \"retest de cada llave sobre las máscaras de referencia\"\n    ]\n  },\n  \"hypotheses\": [\n    {\n      \"id\": \"H-C3\",\n      \"by\": \"Claude (carta 005), congelada por ChatGPT 005\",\n      \"function\": \"pragma_ae.aem1_v14.hypothesis_h_c3\",\n      \"statement\": \"H2 cierra el agujero objetivo de la franja en ≥ 2 de 3 semillas sin empeorar O\",\n      \"holds_if\": \"≥ 2 semillas evaluables con closure = cerrado y sin o_worse\",\n      \"refuted_if\": \"aun contando como éxito las semillas no evaluables, no llegan a 2\",\n      \"otherwise\": \"INDETERMINATE\"\n    },\n    {\n      \"id\": \"H-G5\",\n      \"by\": \"ChatGPT (carta 005)\",\n      \"function\": \"pragma_ae.aem1_v14.hypothesis_h_g5\",\n      \"statement\": \"H2 actúa como reparación local, no como resegmentación global\",\n      \"holds_if\": \"≥ 2/3 semillas cerradas, ninguna con o_worse y ninguna con un agujero D nuevo (pérdida nueva ≥ 1000 px fuera de la región de H2)\",\n      \"refuted_if\": \"≥ 2/3 semillas cerradas CON un agujero D nuevo, o ≥ 2/3 con o_worse\",\n      \"otherwise\": \"INDETERMINATE\",\n      \"role\": \"hipótesis causal y diagnóstica; NO es un quinto requisito de PASS\"\n    }\n  ],\n  \"decision\": {\n    \"acceptance\": \"solo por el protocolo de auditoría v2 rev. 1 §5: los cuatro criterios TRUE en el consenso de las dos llaves, más agujeros, sentinelas y adjudicación. Solo las candidatas con H2 pueden dar PASS. Cerrar la franja no basta: si queda el mentón u otro D, es NO PASS (ChatGPT 005 §6)\",\n    \"case_labels\": {\n      \"PASS_FULL_SUBJECT_UNDER_FIXED_AEM1_PROTOCOL\": \"alguna candidata con H2 pasa\",\n      \"INCONCLUSIVE_SELECTED_OUTPUT_FAILED\": \"ninguna pasa\"\n    },\n    \"stop_rule\": \"A-E(−1) se cierra con v1.4 pase lo que pase: AEM1_CLOSED_DEMONSTRATED o AEM1_CLOSED_INCONCLUSIVE. Otra iteración exige una decisión explícita de las dos IAs y de la persona usuaria (ChatGPT 005)\",\n    \"user_veto\": true,\n    \"sam2_rejectable\": false,\n    \"project_status\": \"INCONCLUSIVE_A_E0_REQUIRED\",\n    \"phase_b\": \"BLOQUEADA\",\n    \"scope_of_a_pass\": \"una foto, un caso, un protocolo fijo: no demuestra A-E1 ni generaliza\"\n  },\n  \"forbidden\": [\n    \"cambiar H2, la caja, los umbrales o cualquier otro prompt después de ver datos de esta corrida\",\n    \"elegir semilla, salida o candidata por score\",\n    \"reparar máscaras (restas, uniones o rellenos): se juzga la salida de SAM 2 tal cual\",\n    \"ejecutar un caso INVALID_PERTURBATION\",\n    \"FastAPI, localhost, YOLO-seg o BiRefNet; tocar la extensión\"\n  ],\n  \"outputs\": {\n    \"zip\": \"PRAGMA_AEM1v14_<run_id>_PENDING_EXTERNAL_AUDIT.zip\",\n    \"contents\": [\n      \"aem1v14_config.json (prerregistro embebido, entorno, compuertas y luma)\",\n      \"aem1v14_calls.json (las llamadas ejecutadas, en orden)\",\n      \"aem1v14_manifest.json (bytes y SHA-256)\",\n      \"aem1v14_report.json\",\n      \"masks/<candidata>.png para BASE, referencia y H2 (9)\",\n      \"aem1v14_perturbaciones.npz (15 máscaras en packbits)\"\n    ],\n    \"not_shown_in_colab\": \"el cuaderno no muestra máscaras, áreas, scores ni sentinelas: solo progreso e integridad\"\n  },\n  \"content_sha256\": \"7048b9fabf03544885d749ad3f19d0dd9d373730d8531f87db58e40ca7220b5d\"\n}\n"
assert hashlib.sha256(PREREG_TEXT.encode("utf-8")).hexdigest() == PREREG_FILE_SHA256, "Prerregistro alterado"
PREREG = json.loads(PREREG_TEXT)
CALL_PLAN = PREREG["call_plan"]
assert PREREG["status"] == "PREREGISTERED" and len(CALL_PLAN) == PREREG["candidate_counts"]["calls"]

gray = image.astype(np.float32).mean(axis=2)   # misma luma que el preflight: media de canales, parche 13×13

def patch_luma(xy, radius=6):
    x, y = int(xy[0]), int(xy[1])
    return float(gray[max(0, y - radius):y + radius + 1, max(0, x - radius):x + radius + 1].mean())

expected_luma = {}
for entry in PREREG["base_config"]["prompts"]:
    expected_luma[tuple(entry["xy"])] = entry["patch_luma_mean"]
for group in PREREG["base_config"]["holdouts"].values():
    for entry in group:
        expected_luma[tuple(entry["xy"])] = entry["patch_luma_mean"]
for entry in PREREG["new_prompts"].values():
    expected_luma[tuple(entry["xy"])] = entry["patch_luma_mean"]
for target in PREREG["branches"]["PERTURB_POINT"]["targets"].values():
    for row in target["perturbations"]:
        expected_luma[tuple(row["xy"])] = row["patch_luma_mean"]
used = {tuple(p) for call in CALL_PLAN for p in call["points"]}
missing = sorted(used - set(expected_luma))
if missing:
    raise RuntimeError(f"FAIL_CONFIG: prompts sin luma registrada: {missing[:5]}")
deviations = {xy: abs(patch_luma(xy) - value) for xy, value in expected_luma.items()}
LUMA_CHECK = {"points_checked": len(deviations), "max_abs_deviation": round(max(deviations.values()), 3), "tolerance": 3.0}
if LUMA_CHECK["max_abs_deviation"] > 3.0:
    raise RuntimeError(f"FAIL_CONFIG: la luma se desvía {LUMA_CHECK['max_abs_deviation']} > 3,0. No se genera nada.")
print("Prerregistro verificado:", PREREG["content_sha256"][:12], "…", "·", len(CALL_PLAN), "llamadas ·",
      "luma de", LUMA_CHECK["points_checked"], "puntos dentro de 3,0")

In [ ]:
# Celda 4 · modelo y un único embedding de la foto (las mismas llamadas que v1.2)
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

MASK_THRESHOLD = 0.0
t0 = time.perf_counter()
try:
    sam2_model = build_sam2(MODEL_CFG, str(CHECKPOINT), device=DEVICE)
    predictor = SAM2ImagePredictor(sam2_model)
    synchronize()
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError("FAIL_ENVIRONMENT: Large no cabe; usa L4 o A100. No se degrada a Small.") from exc
MODEL_LOAD_S = time.perf_counter() - t0

@dataclass
class PromptSet:
    points: np.ndarray
    labels: np.ndarray
    def __post_init__(self):
        self.points = np.asarray(self.points, dtype=np.float32).reshape(-1, 2)
        self.labels = np.asarray(self.labels, dtype=np.int32).reshape(-1)

def generate(prompts=None, box=None, mask_input=None, multimask_output=True):
    synchronize(); start = time.perf_counter()
    with torch.inference_mode(), inference_precision():
        hr_logits, scores, low_res = predictor.predict(
            point_coords=prompts.points if prompts is not None else None,
            point_labels=prompts.labels if prompts is not None else None,
            box=box, mask_input=mask_input, multimask_output=multimask_output, return_logits=True)
    synchronize()
    return (np.asarray(hr_logits, dtype=np.float32) > MASK_THRESHOLD, np.asarray(scores, dtype=np.float32),
            np.asarray(low_res, dtype=np.float32), time.perf_counter() - start)

synchronize(); t0 = time.perf_counter()
with torch.inference_mode(), inference_precision():
    predictor.set_image(image)
synchronize()
EMBEDDING_S = time.perf_counter() - t0
print(f"Modelo listo en {MODEL_LOAD_S:.1f} s · embedding en {EMBEDDING_S:.2f} s")

In [ ]:
# Celda 5 · las 22 llamadas del prerregistro, en orden. Solo se imprime el progreso.
PNG_BRANCHES = ("BASE", "+POS_HAIR+SLEEVE", "+POS_HAIR+SLEEVE+H2")
LOW_RES, EXECUTED, PNG_MASKS, PACKED, PACKED_SHA = {}, [], {}, {}, {}
t_run = time.perf_counter()
for number, call in enumerate(CALL_PLAN, 1):
    prompts = PromptSet(call["points"], call["labels"]) if call["points"] else None
    box = None if call["box"] is None else np.asarray(call["box"], np.float32)
    mask_input = None
    if call["mask_input_from"] is not None:
        source = call["mask_input_from"]
        mask_input = LOW_RES[source["call_id"]][source["index"]][None, :, :]
    masks, scores, low_res, seconds = generate(prompts, box, mask_input, call["multimask_output"])
    assert masks.shape == (len(call["candidates"]), 2248, 4000), masks.shape
    if call["multimask_output"]:
        LOW_RES[call["call_id"]] = low_res          # solo se reutilizan como semillas
    for cid, mask in zip(call["candidates"], masks):
        packed = np.packbits(mask)
        PACKED_SHA[cid] = hashlib.sha256(packed.tobytes()).hexdigest()
        if cid.split("|", 1)[0] in PNG_BRANCHES:
            PNG_MASKS[cid] = mask
        else:
            PACKED[cid] = packed
    EXECUTED.append({**{k: call[k] for k in ("call_id", "branch", "protocol", "points", "labels", "box",
                                               "mask_input_from", "multimask_output", "candidates")},
                     "scores_never_used": [round(float(s), 6) for s in scores], "inference_s": round(seconds, 4)})
    if number % 5 == 0 or number == len(CALL_PLAN):
        print(f"llamada {number}/{len(CALL_PLAN)}")
RUN_S = time.perf_counter() - t_run
assert len(PACKED_SHA) == PREREG["candidate_counts"]["total_masks"]
print(f"Hecho: {len(PACKED_SHA)} máscaras en {RUN_S:.1f} s. No se muestran: la auditoría es ciega.")

In [ ]:
# Celda 6 · manifiesto, ZIP y descarga
def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path

(RUN_DIR / "masks").mkdir(exist_ok=True)
for cid, mask in PNG_MASKS.items():
    Image.fromarray(mask.astype(np.uint8) * 255).save(RUN_DIR / "masks" / (cid.replace("|", "__") + ".png"))
np.savez_compressed(RUN_DIR / "aem1v14_perturbaciones.npz", __shape__=np.array([2248, 4000]),
                    **{cid.replace("|", "__"): packed for cid, packed in PACKED.items()})
write_json(RUN_DIR / "aem1v14_config.json", {
    "notebook_version": "1.4", "run_id": RUN_ID, "prereg_content_sha256": PREREG["content_sha256"],
    "prereg_file_sha256": PREREG_FILE_SHA256, "prereg": PREREG, "environment": ENVIRONMENT, "luma_check": LUMA_CHECK,
    "gates": {"sam2_commit": "verificado en la celda 1", "checkpoint": "verificado en la celda 1",
              "image_sha256": "verificado en la celda 2", "prereg": "verificado en la celda 3", "luma": LUMA_CHECK},
})
write_json(RUN_DIR / "aem1v14_calls.json", EXECUTED)
write_json(RUN_DIR / "aem1v14_report.json", {
    "run_id": RUN_ID, "status": "PENDING_EXTERNAL_AUDIT", "environment": ENVIRONMENT,
    "timings_s": {"model_load": round(MODEL_LOAD_S, 3), "embedding": round(EMBEDDING_S, 3), "calls": round(RUN_S, 3)},
    "gpu_peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3) if DEVICE == "cuda" else None,
    "counts": {"calls": len(EXECUTED), "masks": len(PACKED_SHA), "png": len(PNG_MASKS), "npz": len(PACKED)},
    "note": "sin revisión en Colab: auditoría ciega externa según auditoria/PROTOCOLO_AUDITORIA_AEM1_v2.md (rev. 1)",
})
files_listed = sorted(p for p in RUN_DIR.rglob("*") if p.is_file() and p.name != "P1070614.JPG"
                      and p.name != "aem1v14_manifest.json")
manifest = {"run_id": RUN_ID, "files": {p.relative_to(RUN_DIR).as_posix(): {"bytes": p.stat().st_size, "sha256": sha256_file(p)}
                                        for p in files_listed},
            "masks": dict(sorted(PACKED_SHA.items())), "mask_hash": "sha256(np.packbits(mask)) sin cabecera"}
write_json(RUN_DIR / "aem1v14_manifest.json", manifest)
ZIP_PATH = WORK_DIR / f"PRAGMA_AEM1v14_{RUN_ID}_PENDING_EXTERNAL_AUDIT.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for p in files_listed + [RUN_DIR / "aem1v14_manifest.json"]:
        archive.write(p, p.relative_to(RUN_DIR).as_posix())
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
ZIP_SHA256 = sha256_file(ZIP_PATH)
print(f"ZIP: {ZIP_PATH.name} · {ZIP_PATH.stat().st_size / 2**20:.1f} MiB · SHA-256 {ZIP_SHA256}")
if not PRAGMA_HEADLESS:
    files.download(str(ZIP_PATH))
print("Adjunta este ZIP solo a Claude. No compartas capturas de esta corrida.")

## 7. Qué hacer ahora

1. Busca el ZIP `PRAGMA_AEM1v14_…_PENDING_EXTERNAL_AUDIT.zip` en tu carpeta de descargas.
2. **Adjúntalo a Claude** en la sesión de Claude Code. No se lo mandes a ChatGPT.
3. Claude verifica la integridad sin mirar resultados, prepara el paquete ciego y te lo da. Ese
   paquete es lo único que recibe ChatGPT, sin carta.